# Single FOAR JSON Robot Render All Schemes

这个 notebook 重新按 `ca-pi-xie-device-tools/inspect_json_robot_render_inline_no_side_predefined.ipynb` 的思路写，专门用于 **单臂 FOAR / zihao purple box** 数据。

特点：
- 单臂版本，不再沿用之前那个反复改过的 notebook；
- 真实 depth 点云与单臂 URDF mesh 用同一条变换链渲染；
- 自动发现并渲染 `airexo/airexo/urdf_models/` 下所有 `zihao_single*` 方案，加上 `robot` / `robot_old`；
- 一次性把所有方案都显示出来，不用每次手动切一个；
- 增加桌面平面法向估计与每个方案 base/tcp 轴的夹角诊断。

默认仍然使用：
- scene: `scene_0001`
- frame: `1774521917807`
- arm suffix: `062770`


In [ ]:
from __future__ import annotations

import json
import math
import random
import sys
from collections import OrderedDict
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from PIL import Image
from scipy.spatial.transform import Rotation as R

WORKSPACE_ROOT = Path('/home/haoxiang/rise2_mask_aware')
AIREXO_ROOT = WORKSPACE_ROOT / 'airexo'
URDF_MODELS_ROOT = WORKSPACE_ROOT / 'airexo/airexo/urdf_models'
for p in [WORKSPACE_ROOT, AIREXO_ROOT]:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

from airexo.helpers.constants import (
    O3D_RENDER_TRANSFORMATION,
    ROBOT_PREDEFINED_TRANSFORMATION,
    ROBOT_TCP_TO_FLANGE,
)
from airexo.helpers import urdf_robot as robot_helper


In [ ]:
# ===== 用户参数 =====
RESULT_JSON = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/calib/result.json')
H5_PATH = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/train/scene_0001/lowdim/lowdim.h5')
SCENE_CAM_DIR = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/train/scene_0001/cam_104422070117')

CAMERA_TIMESTAMP = 1774521917807
ARM_SUFFIX = '062770'
EXACT_H5_TIMESTAMP = False
CAM_BASE_MODE = 'base_to_cam__predef'   # 与 ca-pi-xie 默认语义一致

JOINT_SOURCE_MODE = 'h5'   # 'h5' | 'manual_deg'
MANUAL_JOINT_DEG = [0, 0, 0, 0, 0, 0, 0]
MANUAL_GRIPPER_WIDTH = 0.0

SHOW_GRIPPER_LINKS = False
SHOW_CAMERA_FRAME = True
SHOW_BASE_FRAME = True
SHOW_TCP_FRAME = True
SHOW_TABLE_NORMAL = True
FRAME_AXIS_LEN = 0.08
TABLE_NORMAL_LEN = 0.16

DEPTH_SCALE = 1000.0
MIN_DEPTH_M = 0.05
MAX_DEPTH_M = 1.50
POINT_STRIDE = 4
POINT_MAX = 100000
MESH_FACE_LIMIT = 20000
APPLY_O3D_TO_POINT_CLOUD = True

PLANE_RANSAC_ITERS = 600
PLANE_INLIER_THRESH_M = 0.008
PLANE_MAX_POINTS = 20000

SCHEME_INCLUDE_STANDARD = True
SCHEME_PREFIX = 'zihao_single'


In [ ]:
class JointCfg:
    def __init__(self, num_joints=8, num_robot_joints=7):
        self.num_joints = num_joints
        self.num_robot_joints = num_robot_joints

JOINT_CFGS = JointCfg()
GRIPPER_KEYWORDS = ('finger', 'knuckle', 'robotiq')


def invert_T(T):
    T = np.asarray(T, dtype=np.float64)
    out = np.eye(4, dtype=np.float64)
    out[:3, :3] = T[:3, :3].T
    out[:3, 3] = -T[:3, :3].T @ T[:3, 3]
    return out


def pose7_wxyz_to_mat(pose7):
    pose7 = np.asarray(pose7, dtype=np.float64).reshape(7)
    mat = np.eye(4, dtype=np.float64)
    qw, qx, qy, qz = pose7[3:]
    mat[:3, :3] = R.from_quat([qx, qy, qz, qw]).as_matrix()
    mat[:3, 3] = pose7[:3]
    return mat


def load_json_pose_and_intrinsic(path: Path):
    data = json.loads(path.read_text())
    T_base_to_cam = pose7_wxyz_to_mat(data['pose_in_link'])
    intrinsic = np.asarray(data['intrinsics'], dtype=np.float64)
    return T_base_to_cam, intrinsic, data


def cam_base_from_json(T_json, mode):
    robot_predef_inv = invert_T(np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64))
    if mode == 'base_to_cam__predef':
        return invert_T(T_json) @ robot_predef_inv
    if mode == 'base_to_cam__raw':
        return invert_T(T_json)
    if mode == 'cam_to_base__predef':
        return T_json @ robot_predef_inv
    if mode == 'cam_to_base__raw':
        return T_json
    raise ValueError(f'invalid cam_base_mode: {mode}')


def find_h5_index(h5_timestamps: np.ndarray, camera_timestamp: int, exact: bool = False):
    if exact:
        hit = np.where(h5_timestamps == int(camera_timestamp))[0]
        if len(hit) == 0:
            raise ValueError(f'exact timestamp {camera_timestamp} not found in h5')
        return int(hit[0])

    insert_idx = int(np.searchsorted(h5_timestamps, int(camera_timestamp)))
    if insert_idx <= 0:
        return 0
    if insert_idx >= len(h5_timestamps):
        return len(h5_timestamps) - 1
    left_idx = insert_idx - 1
    right_idx = insert_idx
    left_diff = abs(int(h5_timestamps[left_idx]) - int(camera_timestamp))
    right_diff = abs(int(h5_timestamps[right_idx]) - int(camera_timestamp))
    return left_idx if left_diff <= right_diff else right_idx


def extract_joint_and_tcp(h5_path: Path, camera_timestamp: int, arm_suffix: str, exact_ts: bool):
    with h5py.File(h5_path, 'r') as f:
        h5_timestamps = np.asarray(f['timestamp'][:], dtype=np.int64)
        idx = find_h5_index(h5_timestamps, camera_timestamp, exact=exact_ts)
        matched_ts = int(h5_timestamps[idx])
        joint7 = np.asarray(f[f'joint_position_rad_{arm_suffix}'][idx], dtype=np.float64)
        gripper = float(np.asarray(f[f'ee_state_{arm_suffix}'][idx]).reshape(-1)[0])
        tcp = np.asarray(f[f'tcp_pose_{arm_suffix}'][idx], dtype=np.float64)
    joint = np.concatenate([joint7, [gripper]], axis=0)
    return idx, matched_ts, joint, tcp


def manual_joint_from_deg(joint_deg, gripper_width):
    joint_deg = np.asarray(joint_deg, dtype=np.float64).reshape(7)
    return np.concatenate([np.deg2rad(joint_deg), [float(gripper_width)]], axis=0)


def load_scene_images(scene_dir: Path, camera_timestamp: int):
    color_path = scene_dir / 'color' / f'{camera_timestamp}.png'
    depth_path = scene_dir / 'depth' / f'{camera_timestamp}.png'
    color = np.array(Image.open(color_path).convert('RGB'))
    depth = np.array(Image.open(depth_path))
    return color_path, depth_path, color, depth


def depth_to_point_cloud(depth_mm: np.ndarray, intrinsic: np.ndarray, stride: int, depth_scale: float, min_depth_m: float, max_depth_m: float, point_max: int, apply_o3d: bool):
    fx = intrinsic[0, 0]
    fy = intrinsic[1, 1]
    cx = intrinsic[0, 2]
    cy = intrinsic[1, 2]

    depth = depth_mm.astype(np.float64) / depth_scale
    depth = depth[::stride, ::stride]
    h, w = depth.shape
    ys, xs = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    xs = xs * stride
    ys = ys * stride

    valid = np.isfinite(depth) & (depth > min_depth_m) & (depth < max_depth_m)
    z = depth[valid]
    x = (xs[valid] - cx) * z / fx
    y = (ys[valid] - cy) * z / fy
    pts = np.stack([x, y, z], axis=1)

    if apply_o3d:
        pts_h = np.concatenate([pts, np.ones((pts.shape[0], 1), dtype=np.float64)], axis=1)
        pts = (np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64) @ pts_h.T).T[:, :3]

    if pts.shape[0] > point_max:
        idx = np.linspace(0, pts.shape[0] - 1, point_max).astype(np.int64)
        pts = pts[idx]
    return pts


def should_keep_link(link_name: str):
    if SHOW_GRIPPER_LINKS:
        return True
    return not any(keyword in str(link_name).lower() for keyword in GRIPPER_KEYWORDS)


def resolve_mesh_path(mesh_rel_path: str, urdf_file: str):
    urdf_parent = Path(urdf_file).parent
    candidates = [
        urdf_parent / mesh_rel_path,
        URDF_MODELS_ROOT / 'robot' / mesh_rel_path,
        URDF_MODELS_ROOT / 'robot_old' / mesh_rel_path,
    ]
    for cand in candidates:
        if cand.exists():
            return cand
    raise FileNotFoundError(f'cannot resolve mesh path: {mesh_rel_path} from urdf={urdf_file}')


def load_mesh_vertices_faces(mesh_rel_path: str, urdf_file: str):
    import trimesh
    mesh_path = resolve_mesh_path(mesh_rel_path, urdf_file)
    mesh = trimesh.load_mesh(mesh_path, process=False)
    if hasattr(mesh, 'geometry'):
        mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))
    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    faces = np.asarray(mesh.faces, dtype=np.int32)
    return vertices, faces


def apply_transform(vertices: np.ndarray, T: np.ndarray):
    homo = np.concatenate([vertices, np.ones((vertices.shape[0], 1), dtype=np.float64)], axis=1)
    out = (T @ homo.T).T
    return out[:, :3]


def downsample_faces(faces: np.ndarray, limit: int):
    if faces.shape[0] <= limit:
        return faces
    idx = np.linspace(0, faces.shape[0] - 1, limit).astype(np.int64)
    return faces[idx]


def add_frame(fig, T, name, axis_len=0.08):
    origin = T[:3, 3]
    axes = T[:3, :3]
    colors = ['red', 'green', 'blue']
    labels = ['x', 'y', 'z']
    for i in range(3):
        p1 = origin
        p2 = origin + axes[:, i] * axis_len
        fig.add_trace(go.Scatter3d(
            x=[p1[0], p2[0]],
            y=[p1[1], p2[1]],
            z=[p1[2], p2[2]],
            mode='lines',
            line=dict(color=colors[i], width=7),
            name=f'{name}_{labels[i]}',
            showlegend=False,
        ))


def add_vector(fig, origin, vector, name, color='magenta'):
    p1 = np.asarray(origin, dtype=np.float64).reshape(3)
    p2 = p1 + np.asarray(vector, dtype=np.float64).reshape(3)
    fig.add_trace(go.Scatter3d(
        x=[p1[0], p2[0]],
        y=[p1[1], p2[1]],
        z=[p1[2], p2[2]],
        mode='lines',
        line=dict(color=color, width=8),
        name=name,
        showlegend=False,
    ))


def safe_link_tf(tf_map, key):
    if key not in tf_map:
        return None
    return np.asarray(tf_map[key].matrix(), dtype=np.float64)


def fk_tcp_candidates(joint, joint_cfgs, urdf_file):
    tf_map = robot_helper.forward_kinematic_single(
        joint=joint.astype(np.float32),
        joint_cfgs=joint_cfgs,
        is_rad=True,
        urdf_file=urdf_file,
        with_visuals_map=False,
    )
    cand = {}
    flange = safe_link_tf(tf_map, 'flange')
    link7 = safe_link_tf(tf_map, 'link7')
    if flange is not None:
        cand['flange'] = flange
        cand['tcp_from_flange'] = flange @ invert_T(np.asarray(ROBOT_TCP_TO_FLANGE, dtype=np.float64))
    if link7 is not None:
        cand['link7'] = link7
    return cand


def compose_world_tf(cam_to_base, local_tf):
    return (
        np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
        @ np.asarray(cam_to_base, dtype=np.float64)
        @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
        @ np.asarray(local_tf, dtype=np.float64)
    )


def compose_mesh_tf(cam_to_base, transform, offset):
    return compose_world_tf(cam_to_base, transform.matrix() @ offset.matrix())


def compose_real_base_world(T_base_to_cam):
    return np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64) @ invert_T(T_base_to_cam)


def compose_real_tcp_world(T_base_to_cam, tcp_pose7):
    return compose_real_base_world(T_base_to_cam) @ pose7_wxyz_to_mat(tcp_pose7)


def angle_deg_to_normal(vec, normal):
    v = np.asarray(vec, dtype=np.float64).reshape(3)
    n = np.asarray(normal, dtype=np.float64).reshape(3)
    v = v / max(np.linalg.norm(v), 1e-12)
    n = n / max(np.linalg.norm(n), 1e-12)
    cosv = float(np.clip(abs(np.dot(v, n)), 0.0, 1.0))
    return math.degrees(math.acos(cosv))


def discover_scheme_urdfs():
    schemes = OrderedDict()
    if SCHEME_INCLUDE_STANDARD:
        schemes['robot'] = URDF_MODELS_ROOT / 'robot' / 'left_robot.urdf'
        schemes['robot_old'] = URDF_MODELS_ROOT / 'robot_old' / 'left_robot.urdf'
    for path in sorted(URDF_MODELS_ROOT.glob(f'{SCHEME_PREFIX}*/left_robot.urdf')):
        schemes[path.parent.name] = path
    return schemes


def fit_plane_ransac(points: np.ndarray, max_points: int, iters: int, dist_thresh: float):
    pts = np.asarray(points, dtype=np.float64)
    if pts.shape[0] < 3:
        raise ValueError('not enough points for plane fitting')
    if pts.shape[0] > max_points:
        idx = np.linspace(0, pts.shape[0] - 1, max_points).astype(np.int64)
        pts = pts[idx]

    rng = np.random.default_rng(233)
    best = None
    best_count = -1
    for _ in range(int(iters)):
        ids = rng.choice(pts.shape[0], size=3, replace=False)
        p0, p1, p2 = pts[ids]
        n = np.cross(p1 - p0, p2 - p0)
        nn = np.linalg.norm(n)
        if nn < 1e-9:
            continue
        n = n / nn
        d = -float(np.dot(n, p0))
        dist = np.abs(pts @ n + d)
        inliers = dist < float(dist_thresh)
        count = int(inliers.sum())
        if count > best_count:
            best_count = count
            best = (n.copy(), d, inliers)

    if best is None:
        raise RuntimeError('plane fitting failed')

    n, d, inliers = best
    inlier_pts = pts[inliers]
    center = inlier_pts.mean(axis=0)
    cov = np.cov((inlier_pts - center).T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    normal = eigvecs[:, int(np.argmin(eigvals))]
    normal = normal / max(np.linalg.norm(normal), 1e-12)
    if normal[2] < 0:
        normal = -normal
    return {
        'normal': normal,
        'center': center,
        'inlier_count': int(inliers.sum()),
        'sample_count': int(pts.shape[0]),
    }


In [ ]:
T_base_to_cam, intrinsic, raw_json = load_json_pose_and_intrinsic(RESULT_JSON)
cam_to_base = cam_base_from_json(T_base_to_cam, CAM_BASE_MODE)

idx, matched_ts, joint_h5, tcp_h5 = extract_joint_and_tcp(
    H5_PATH, CAMERA_TIMESTAMP, ARM_SUFFIX, EXACT_H5_TIMESTAMP
)

if JOINT_SOURCE_MODE == 'manual_deg':
    joint_used = manual_joint_from_deg(MANUAL_JOINT_DEG, MANUAL_GRIPPER_WIDTH)
else:
    joint_used = joint_h5.copy()

color_path, depth_path, color_img, depth_img = load_scene_images(SCENE_CAM_DIR, CAMERA_TIMESTAMP)
point_cloud = depth_to_point_cloud(
    depth_img,
    intrinsic,
    stride=POINT_STRIDE,
    depth_scale=DEPTH_SCALE,
    min_depth_m=MIN_DEPTH_M,
    max_depth_m=MAX_DEPTH_M,
    point_max=POINT_MAX,
    apply_o3d=APPLY_O3D_TO_POINT_CLOUD,
)
plane = fit_plane_ransac(
    point_cloud,
    max_points=PLANE_MAX_POINTS,
    iters=PLANE_RANSAC_ITERS,
    dist_thresh=PLANE_INLIER_THRESH_M,
)

print('camera_timestamp       =', CAMERA_TIMESTAMP)
print('matched_h5_index       =', idx)
print('matched_h5_timestamp   =', matched_ts)
print('timestamp_diff_ms      =', abs(matched_ts - CAMERA_TIMESTAMP))
print('cam_base_mode          =', CAM_BASE_MODE)
print('point_cloud shape      =', point_cloud.shape)
print('plane sample_count     =', plane['sample_count'])
print('plane inlier_count     =', plane['inlier_count'])
print('plane normal           =', plane['normal'])
print('plane center           =', plane['center'])
print('joint_used             =', joint_used)
print('tcp_h5                 =', tcp_h5)
print('json parent            =', raw_json.get('parent_link_name'))
print('json cam_serial        =', raw_json.get('cam_serial'))
print('T_base_to_cam =')
print(T_base_to_cam)
print('cam_to_base(eval style) =')
print(cam_to_base)
display(Image.open(color_path))


In [ ]:
SCHEME_URDFS = discover_scheme_urdfs()
print('scheme_count =', len(SCHEME_URDFS))
for key, path in SCHEME_URDFS.items():
    print(key, '->', path)


In [ ]:
def collect_scheme_info(name: str, urdf_path: Path):
    tf_map, visuals_map = robot_helper.forward_kinematic_single(
        joint=joint_used.astype(np.float32),
        joint_cfgs=JOINT_CFGS,
        is_rad=True,
        urdf_file=str(urdf_path),
        with_visuals_map=True,
    )
    base_link_tf = safe_link_tf(tf_map, 'base_link')
    arm_root_tf = safe_link_tf(tf_map, 'arm_root_link')
    fk_cand = fk_tcp_candidates(joint_used, JOINT_CFGS, str(urdf_path))

    base_world = None if base_link_tf is None else compose_world_tf(cam_to_base, base_link_tf)
    arm_root_world = None if arm_root_tf is None else compose_world_tf(cam_to_base, arm_root_tf)
    fk_tcp_world = None if 'tcp_from_flange' not in fk_cand else compose_world_tf(cam_to_base, fk_cand['tcp_from_flange'])
    h5_tcp_world = compose_real_tcp_world(T_base_to_cam, tcp_h5)

    row = {
        'scheme': name,
        'urdf': str(urdf_path),
        'has_base_link': base_link_tf is not None,
        'has_arm_root_link': arm_root_tf is not None,
        'num_links': len(tf_map),
        'num_visual_links': len(visuals_map),
    }

    if base_world is not None:
        row['base_x_to_table_normal_deg'] = round(angle_deg_to_normal(base_world[:3, 0], plane['normal']), 3)
        row['base_y_to_table_normal_deg'] = round(angle_deg_to_normal(base_world[:3, 1], plane['normal']), 3)
        row['base_z_to_table_normal_deg'] = round(angle_deg_to_normal(base_world[:3, 2], plane['normal']), 3)
    else:
        row['base_x_to_table_normal_deg'] = np.nan
        row['base_y_to_table_normal_deg'] = np.nan
        row['base_z_to_table_normal_deg'] = np.nan

    if fk_tcp_world is not None:
        row['fk_tcp_x_to_table_normal_deg'] = round(angle_deg_to_normal(fk_tcp_world[:3, 0], plane['normal']), 3)
        row['fk_tcp_y_to_table_normal_deg'] = round(angle_deg_to_normal(fk_tcp_world[:3, 1], plane['normal']), 3)
        row['fk_tcp_z_to_table_normal_deg'] = round(angle_deg_to_normal(fk_tcp_world[:3, 2], plane['normal']), 3)
        row['fk_tcp_pos_err_m'] = round(float(np.linalg.norm(fk_tcp_world[:3, 3] - h5_tcp_world[:3, 3])), 4)
    else:
        row['fk_tcp_x_to_table_normal_deg'] = np.nan
        row['fk_tcp_y_to_table_normal_deg'] = np.nan
        row['fk_tcp_z_to_table_normal_deg'] = np.nan
        row['fk_tcp_pos_err_m'] = np.nan

    return {
        'name': name,
        'urdf_path': urdf_path,
        'tf_map': tf_map,
        'visuals_map': visuals_map,
        'base_world': base_world,
        'arm_root_world': arm_root_world,
        'fk_cand': fk_cand,
        'fk_tcp_world': fk_tcp_world,
        'row': row,
    }


SCHEME_INFOS = [collect_scheme_info(name, path) for name, path in SCHEME_URDFS.items()]
summary_df = pd.DataFrame([x['row'] for x in SCHEME_INFOS])
summary_df = summary_df.sort_values(
    by=['base_x_to_table_normal_deg', 'base_y_to_table_normal_deg', 'base_z_to_table_normal_deg'],
    kind='stable',
).reset_index(drop=True)
display(summary_df)


In [ ]:
def build_scheme_figure(info):
    fig = go.Figure()

    if point_cloud.shape[0] > 0:
        fig.add_trace(go.Scatter3d(
            x=point_cloud[:, 0],
            y=point_cloud[:, 1],
            z=point_cloud[:, 2],
            mode='markers',
            marker=dict(size=1.0, color=point_cloud[:, 2], colorscale='Viridis', opacity=0.28),
            name='depth_point_cloud',
            showlegend=False,
        ))

    for link, transform in info['tf_map'].items():
        if not should_keep_link(link):
            continue
        for v in info['visuals_map'][link]:
            if v.geom_param is None:
                continue
            verts, faces = load_mesh_vertices_faces(v.geom_param, str(info['urdf_path']))
            faces = downsample_faces(faces, MESH_FACE_LIMIT)
            tf = compose_mesh_tf(cam_to_base, transform, v.offset)
            verts_tf = apply_transform(verts, tf)
            fig.add_trace(go.Mesh3d(
                x=verts_tf[:, 0],
                y=verts_tf[:, 1],
                z=verts_tf[:, 2],
                i=faces[:, 0],
                j=faces[:, 1],
                k=faces[:, 2],
                color='rgba(80,160,255,0.58)',
                opacity=0.55,
                name=f"{info['name']}::{link}",
                showscale=False,
                hoverinfo='name',
            ))

    if SHOW_CAMERA_FRAME:
        add_frame(fig, np.eye(4), 'camera', axis_len=FRAME_AXIS_LEN)

    if SHOW_TABLE_NORMAL:
        add_vector(fig, plane['center'], plane['normal'] * TABLE_NORMAL_LEN, 'table_normal', color='magenta')

    if SHOW_BASE_FRAME and info['base_world'] is not None:
        add_frame(fig, info['base_world'], 'base_link', axis_len=FRAME_AXIS_LEN)

    if SHOW_TCP_FRAME:
        add_frame(fig, compose_real_tcp_world(T_base_to_cam, tcp_h5), 'h5_tcp', axis_len=FRAME_AXIS_LEN * 0.8)
        if info['fk_tcp_world'] is not None:
            add_frame(fig, info['fk_tcp_world'], 'fk_tcp', axis_len=FRAME_AXIS_LEN * 0.8)

    angle_bits = []
    for axis_name in ['x', 'y', 'z']:
        key = f'base_{axis_name}_to_table_normal_deg'
        if pd.notna(info['row'][key]):
            angle_bits.append(f"base_{axis_name}:{info['row'][key]:.1f}deg")

    fig.update_layout(
        title=f"{info['name']} | " + ' | '.join(angle_bits),
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data',
        ),
        width=1280,
        height=920,
        showlegend=False,
    )
    return fig


for info in SCHEME_INFOS:
    print('=' * 120)
    print('scheme =', info['name'])
    print('urdf   =', info['urdf_path'])
    print('base angles to table normal =', info['row']['base_x_to_table_normal_deg'], info['row']['base_y_to_table_normal_deg'], info['row']['base_z_to_table_normal_deg'])
    print('fk_tcp_pos_err_m            =', info['row']['fk_tcp_pos_err_m'])
    if info['base_world'] is not None:
        print('base_world rotation columns =')
        print(info['base_world'][:3, :3])
    display(build_scheme_figure(info))
